# 04. The Quantum Harmonic Oscillator: Continuous Waves to Ladder Operators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_0_qm_fundamentals/04_quantum_harmonic_oscillator.ipynb)

* **Module:** 0 — Quantum Mechanics Fundamentals
* **Difficulty:** Intermediate
* **Estimated Time:** 45 minutes
* **Prerequisites:** Notebook 0.1 (Continuous Grids), Notebook 0.2 (Dirac Notation), Notebook 0.3 (Operators & Commutators).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Solve the continuous stationary Schrödinger equation** for the harmonic oscillator by diagonalizing the finite-difference Hamiltonian matrix using `scipy.linalg.eigh`.
2. **Benchmark numerical wavefunctions** $\psi_n(x)$ against analytical Hermite-Gauss polynomials and verify the equispaced energy spectrum $E_n = \hbar\omega(n + 1/2)$.
3. **Formulate the algebraic ladder operator method** in QuTiP using creation ($\hat{a}^\dagger$) and annihilation ($\hat{a}$) operators.
4. **Bridge mechanical oscillators to quantum optics:** prove that a single mode of the quantized electromagnetic field is mathematically equivalent to a harmonic oscillator whose excitations are **photons**.
5. **Evaluate dipole transition matrix elements** $\langle m | \hat{x} | n \rangle$ and verify the optical selection rule $\Delta n = \pm 1$.
6. **Observe the Bohr Correspondence Principle** at higher quantum numbers ($n \gg 1$), where quantum probability densities peak near the classical turning points.


---
## 0. Setup & Dependencies

We will use NumPy, SciPy (for Hermite polynomials and linear algebra), Matplotlib, and QuTiP.
Throughout this notebook, we operate in dimensionless oscillator units where $\hbar = 1$, mass $m = 1$, and angular frequency $\omega = 1$.


In [ ]:
# Install dependencies in Google Colab (uncomment if running in Colab)
# !pip install -q qutip numpy scipy matplotlib ipywidgets

import numpy as np
import scipy.linalg as la
import scipy.special as special
import math
import scipy.integrate as integrate
import matplotlib.pyplot as plt

# QuTiP import
try:
    import qutip as qt
    from qutip import destroy, create, num, basis
    print(f"QuTiP version {qt.__version__} initialized.")
except ImportError:
    print("QuTiP not installed locally. QuTiP cells will use fallback or require pip install qutip.")

# Natural oscillator units
HBAR = 1.0
MASS = 1.0
OMEGA = 1.0

# Configure clean, publication-ready plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

print(f"Oscillator parameters: m = {MASS}, omega = {OMEGA}, hbar = {HBAR}")


---
## 1. The Continuous Harmonic Oscillator: Finite-Difference Solver

The classical simple harmonic oscillator experiences a Hooke's law restoring force $F = -kx = -m\omega^2 x$, corresponding to the quadratic potential energy $V(x) = \frac{1}{2}m\omega^2 x^2$.

The time-independent Schrödinger equation (TISE) is:

$$\hat{H}\psi_n(x) = \left( -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + \frac{1}{2}m\omega^2 x^2 \right) \psi_n(x) = E_n \psi_n(x)$$

### Finite-Difference Matrix Discretization
To solve this boundary-value differential equation numerically on a spatial grid $x_0, x_1, \dots, x_{N-1}$ with spacing $\Delta x$:

1. **Second Derivative (Kinetic Energy):** Using the 3-point central finite-difference stencil:
   $$\frac{d^2\psi}{dx^2}\Big|_{x_i} \approx \frac{\psi_{i+1} - 2\psi_i + \psi_{i-1}}{\Delta x^2}$$
   The kinetic energy operator $\hat{T} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2}$ becomes a symmetric tridiagonal matrix:
   $$\hat{T}_{i,i} = \frac{\hbar^2}{m\Delta x^2}, \quad \hat{T}_{i, i\pm 1} = -\frac{\hbar^2}{2m\Delta x^2}$$

2. **Potential Energy:** The potential $\hat{V}$ is simply the diagonal matrix:
   $$\hat{V}_{i,i} = \frac{1}{2}m\omega^2 x_i^2$$

The complete Hamiltonian matrix is $\hat{H} = \hat{T} + \hat{V}$. Diagonalizing this real symmetric matrix using `scipy.linalg.eigh` immediately yields all energy eigenvalues $E_n$ and discrete eigenvectors $\vec{\psi}_n$.


In [ ]:
def solve_continuous_harmonic_oscillator(L: float = 6.0, N: int = 500):
    # Construct 1D spatial grid
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    
    # Kinetic energy matrix (symmetric tridiagonal)
    diag_T = (HBAR**2 / (MASS * dx**2)) * np.ones(N)
    off_diag_T = (-HBAR**2 / (2.0 * MASS * dx**2)) * np.ones(N - 1)
    T_mat = np.diag(diag_T) + np.diag(off_diag_T, k=1) + np.diag(off_diag_T, k=-1)
    
    # Potential energy matrix (diagonal)
    V_vec = 0.5 * MASS * (OMEGA**2) * (x**2)
    V_mat = np.diag(V_vec)
    
    # Total Hamiltonian matrix
    H_mat = T_mat + V_mat
    
    # Solve eigenvalue problem for real symmetric matrix
    eigenvalues, eigenvectors = la.eigh(H_mat)
    
    # Normalize wavefunctions: \int |psi|^2 dx = 1
    # Each column of eigenvectors is an unscaled eigenvector
    normalized_wavefunctions = []
    for k in range(len(eigenvalues)):
        psi_k = eigenvectors[:, k]
        norm = np.sqrt(integrate.simpson(y=psi_k**2, x=x))
        # Ensure consistent phase convention: positive at first major antinode
        if psi_k[np.argmax(np.abs(psi_k))] < 0:
            psi_k = -psi_k
        normalized_wavefunctions.append(psi_k / norm)
        
    return x, eigenvalues, np.array(normalized_wavefunctions), V_vec

# Solve for grid [-6, 6] with N = 600 points
x_grid, energies_num, wavefuncs_num, V_potential = solve_continuous_harmonic_oscillator(L=6.0, N=600)

print("=== First 6 Numerical Energy Eigenvalues vs Theoretical En = (n + 1/2) ===")
for n in range(6):
    E_exact = (n + 0.5) * HBAR * OMEGA
    E_num = energies_num[n]
    print(f"Level n = {n}: Numerical En = {E_num:.6f} | Exact En = {E_exact:.6f} | Error = {abs(E_num - E_exact):.2e}")


---
## 2. Analytical Hermite-Gauss Eigenfunctions & Verification

The exact analytical solution of the harmonic oscillator stationary Schrödinger equation is:

$$\psi_n(x) = \left(\frac{m\omega}{\pi\hbar}\right)^{1/4} \frac{1}{\sqrt{2^n n!}} \, H_n\left(\sqrt{\frac{m\omega}{\hbar}}x\right) \exp\left( -\frac{m\omega x^2}{2\hbar} \right)$$

where $H_n(\xi)$ are the **Hermite polynomials**:
* $H_0(\xi) = 1$
* $H_1(\xi) = 2\xi$
* $H_2(\xi) = 4\xi^2 - 2$
* $H_3(\xi) = 8\xi^3 - 12\xi$
* $H_4(\xi) = 16\xi^4 - 48\xi^2 + 12$

Let us compare our numerical finite-difference eigenvectors against the exact analytical Hermite-Gauss functions via `scipy.special.hermite`.


In [ ]:
def analytical_harmonic_eigenfunction(n: int, x: np.ndarray) -> np.ndarray:
    xi = np.sqrt(MASS * OMEGA / HBAR) * x
    prefactor = (MASS * OMEGA / (np.pi * HBAR))**0.25 / np.sqrt(2**n * math.factorial(n))
    # Evaluate Physicists' Hermite polynomial H_n(xi)
    Hn = special.eval_hermite(n, xi)
    gaussian = np.exp(-0.5 * xi**2)
    return prefactor * Hn * gaussian

# Plot Energy Level Diagram with superimposed wavefunctions
fig, ax = plt.subplots(figsize=(10, 7))

# Plot the parabolic potential well
ax.plot(x_grid, V_potential, 'k-', lw=2.2, label=r"Potential $V(x) = \frac{1}{2}m\omega^2 x^2$")

# Plot lowest 5 energy levels and wavefunctions shifted by their energy En
scale_factor = 0.55  # visual scaling for wavefunction amplitudes
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

for n in range(5):
    En = energies_num[n]
    psi_exact = analytical_harmonic_eigenfunction(n, x_grid)
    psi_numeric = wavefuncs_num[n]
    
    # Horizontal energy line
    ax.axhline(En, color="gray", ls=":", alpha=0.6)
    
    # Plot numerical vs analytical wavefunctions shifted to energy level En
    ax.plot(x_grid, En + scale_factor * psi_numeric, color=colors[n], lw=2.0,
            label=rf"$n={n}$ ($E_{n}={En:.1f}\hbar\omega$)")
    ax.plot(x_grid, En + scale_factor * psi_exact, color="black", ls="--", lw=1.2, alpha=0.7)
    
    # Classical turning points: V(x_tp) = En -> x_tp = sqrt(2*En / (m*omega^2))
    x_tp = np.sqrt(2.0 * En / (MASS * OMEGA**2))
    ax.plot([-x_tp, x_tp], [En, En], 'o', color=colors[n], markersize=5)

ax.set_title(r"Harmonic Oscillator: Energy Spectrum and Wavefunctions $\psi_n(x)$")
ax.set_xlabel("Position Coordinate $x$")
ax.set_ylabel("Energy $E$ (dimensionless)")
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(0, 6.0)
ax.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

# Quantitative Fidelity Check: Overlap integral <psi_num | psi_exact>
print("=== Overlap Fidelity <psi_numerical | psi_exact> ===")
for n in range(5):
    psi_exact = analytical_harmonic_eigenfunction(n, x_grid)
    fidelity = integrate.simpson(y=wavefuncs_num[n] * psi_exact, x=x_grid)
    print(f"State n = {n}: Overlap = {fidelity:.8f} (Exact = 1.00000000)")


> **Physical Insight:** 
> 1. **Zero-Point Energy ($E_0 = \frac{1}{2}\hbar\omega$):** Even in the ground state ($n=0$), the particle possesses non-zero kinetic and potential energy! This irreducible zero-point motion is mandated by the Heisenberg uncertainty principle ($\Delta x \Delta p \ge \hbar/2$).
> 2. **Quantum Tunneling:** Notice that the wavefunctions penetrate into the classically forbidden regions $|x| > x_{\text{tp}}$ beyond the classical turning points (marked with dots), decaying exponentially into the barrier.


---
## 3. The Algebraic Ladder Operator Formulation (QuTiP)

In 1925, Paul Dirac discovered that one can solve the harmonic oscillator without solving differential equations at all, using **operator algebra**.

We define two non-Hermitian dimensionless operators called **ladder operators**:

$$\hat{a} = \sqrt{\frac{m\omega}{2\hbar}}\left(\hat{x} + \frac{i}{m\omega}\hat{p}\right) \quad (\text{Annihilation / Lowering Operator})$$

$$\hat{a}^\dagger = \sqrt{\frac{m\omega}{2\hbar}}\left(\hat{x} - \frac{i}{m\omega}\hat{p}\right) \quad (\text{Creation / Raising Operator})$$

### The Canonical Bosonic Commutation Relation
Using $[\hat{x}, \hat{p}] = i\hbar$, the commutator between $\hat{a}$ and $\hat{a}^\dagger$ is:

$$[\hat{a}, \hat{a}^\dagger] = \hat{a}\hat{a}^\dagger - \hat{a}^\dagger\hat{a} = 1$$

### The Number Operator & Hamiltonian
The **photon number operator** (or excitation number operator) is defined as:

$$\hat{n} \equiv \hat{a}^\dagger \hat{a}$$

Expressing $\hat{x}$ and $\hat{p}$ in terms of ladder operators:
$$\hat{x} = \sqrt{\frac{\hbar}{2m\omega}}\left(\hat{a} + \hat{a}^\dagger\right), \quad \hat{p} = -i\sqrt{\frac{m\hbar\omega}{2}}\left(\hat{a} - \hat{a}^\dagger\right)$$

Substituting into $\hat{H} = \frac{\hat{p}^2}{2m} + \frac{1}{2}m\omega^2 \hat{x}^2$ yields:

$$\hat{H} = \hbar\omega \left(\hat{a}^\dagger \hat{a} + \frac{1}{2}\right) = \hbar\omega \left(\hat{n} + \frac{1}{2}\right)$$

### Action on Fock States $|n\rangle$
The energy eigenstates $|n\rangle$ (called **Fock states** or **number states**) satisfy:

$$\hat{n}|n\rangle = n|n\rangle \quad (n = 0, 1, 2, \dots)$$

$$\hat{a}|n\rangle = \sqrt{n}|n-1\rangle \quad (\text{with } \hat{a}|0\rangle = 0)$$

$$\hat{a}^\dagger|n\rangle = \sqrt{n+1}|n+1\rangle$$

In QuTiP, discrete Fock spaces are modeled by truncating the infinite Hilbert space to a cutoff dimension $N$:
* `destroy(N)` creates the $N \times N$ annihilation operator $\hat{a}$.
* `create(N)` creates the $N \times N$ creation operator $\hat{a}^\dagger$.
* `num(N)` creates the $N \times N$ number operator $\hat{n}$.


In [ ]:
# Truncated Fock space cutoff dimension
N_cutoff = 8

try:
    # Generate ladder operators in QuTiP
    a = destroy(N_cutoff)
    a_dag = create(N_cutoff)
    n_op = num(N_cutoff)
    H_osc = HBAR * OMEGA * (a_dag * a + 0.5 * qt.qeye(N_cutoff))
    a_mat = a.full()
    adag_mat = a_dag.full()
    H_mat_fock = H_osc.full()
    comm_a = (a * a_dag - a_dag * a).full()
except Exception:
    # Pure NumPy fallback
    a_mat = np.zeros((N_cutoff, N_cutoff))
    for k in range(1, N_cutoff):
        a_mat[k-1, k] = np.sqrt(k)
    adag_mat = a_mat.T
    H_mat_fock = HBAR * OMEGA * (adag_mat @ a_mat + 0.5 * np.eye(N_cutoff))
    comm_a = a_mat @ adag_mat - adag_mat @ a_mat

print("Annihilation Operator a (Tridiagonal Superdiagonal sqrt(n)):")
print(np.round(a_mat, 2))

print("\nCreation Operator a_dag (Tridiagonal Subdiagonal sqrt(n+1)):")
print(np.round(adag_mat, 2))

print("\nHamiltonian Matrix H (Diagonal with En = n + 0.5):")
print(np.diag(H_mat_fock))

print("\nCommutator [a, a_dag] diagonal (all 1.0 except boundary):")
print(np.diag(comm_a))


---
## 4. The Fundamental Bridge to Quantum Optics: Field Quantization

Why is the quantum harmonic oscillator the central pillar of quantum optics?

When James Clerk Maxwell's classical electromagnetic equations are placed inside a reflecting cavity of volume $V$, the vector potential $\mathbf{A}(\mathbf{r}, t)$ can be expanded into spatial cavity modes:

$$\mathbf{A}(\mathbf{r}, t) = \sum_k q_k(t) \mathbf{u}_k(\mathbf{r})$$

Substituting this expansion into the electromagnetic field energy:

$$\mathcal{H}_{\text{EM}} = \frac{1}{2}\int \left( \epsilon_0 \mathbf{E}^2 + \frac{1}{\mu_0}\mathbf{B}^2 \right) d^3r = \sum_k \left( \frac{p_k^2}{2} + \frac{1}{2}\omega_k^2 q_k^2 \right)$$

Each independent electromagnetic cavity mode behaves **identically to a simple harmonic oscillator** with generalized coordinate $q_k$ and momentum $p_k$!

### The Mapping: Mechanical Oscillator $\iff$ Optical Cavity Mode

| Mechanical Harmonic Oscillator | Quantized Optical Cavity Mode |
| :--- | :--- |
| Coordinate $\hat{x}$ | Electric Field Quadrature $\hat{X} \propto \frac{1}{2}(\hat{a} + \hat{a}^\dagger)$ |
| Momentum $\hat{p}$ | Magnetic Field Quadrature $\hat{P} \propto \frac{1}{2i}(\hat{a} - \hat{a}^\dagger)$ |
| Mass $m$ and Frequency $\omega$ | Mode cavity frequency $\omega_k$ |
| Energy Quanta $\hbar\omega$ | **Photons** |
| Ground state $|0\rangle$ (Zero-point motion) | **Electromagnetic Vacuum** (Vacuum fluctuations) |
| Position uncertainty $\Delta x_0 = \sqrt{\frac{\hbar}{2m\omega}}$ | Vacuum field fluctuations $\mathcal{E}_0 = \sqrt{\frac{\hbar\omega}{2\epsilon_0 V}}$ |

### Physical Consequences of the Vacuum
Even in complete darkness (zero photons, $|0\rangle$), the electric field does not vanish! Its expectation value is zero ($\langle 0 | \hat{E} | 0 \rangle = 0$), but its variance is non-zero:

$$\langle 0 | \hat{E}^2 | 0 \rangle = \mathcal{E}_0^2 \ne 0$$

These vacuum fluctuations are responsible for:
1. **Spontaneous emission** (an excited atom is stimulated to emit by vacuum fluctuations).
2. The **Casimir effect** (measurable attractive force between conducting plates in vacuum).
3. The **Lamb shift** in hydrogen.


In [ ]:
# Visualizing the Quadratures of Light as Harmonic Oscillator Variables
def plot_quadrature_distributions():
    # Probability distribution of measuring electric field quadrature X:
    # P(X) = |psi_n(x)|^2
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # 1. Electric Field (Coordinate X) Distributions
    ax1.plot(x_grid, np.abs(wavefuncs_num[0])**2, color="#1f77b4", lw=2, label=r"Vacuum $|0\rangle$ (Ground State)")
    ax1.plot(x_grid, np.abs(wavefuncs_num[1])**2, color="#ff7f0e", lw=2, label=r"Single Photon $|1\rangle$ (1 Node)")
    ax1.plot(x_grid, np.abs(wavefuncs_num[2])**2, color="#2ca02c", lw=2, label=r"Two Photons $|2\rangle$ (2 Nodes)")
    ax1.fill_between(x_grid, np.abs(wavefuncs_num[0])**2, color="#1f77b4", alpha=0.2, label=r"Vacuum Fluctuations Area")
    ax1.set_title("Optical Field Quadrature Distributions $P(X) \propto P(E)$")
    ax1.set_xlabel("Field Quadrature $X$ (Electric Field)")
    ax1.set_ylabel("Probability Density")
    ax1.set_xlim(-4, 4)
    ax1.legend(loc="upper right")
    
    # 2. Number State Photon Probabilities
    fock_indices = np.arange(6)
    probs_vac = [1.0] + [0.0]*5
    probs_p1 = [0.0, 1.0] + [0.0]*4
    
    ax2.bar(fock_indices - 0.15, probs_vac, width=0.3, color="#1f77b4", label=r"Vacuum $|0\rangle$")
    ax2.bar(fock_indices + 0.15, probs_p1, width=0.3, color="#ff7f0e", label=r"Single Photon $|1\rangle$")
    ax2.set_xticks(fock_indices)
    ax2.set_xlabel("Photon Number $n$")
    ax2.set_ylabel("Probability $P(n)$")
    ax2.set_title("Photon Number Statistics (Discrete Fock States)")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

plot_quadrature_distributions()


---
## 5. Transition Matrix Elements & Optical Selection Rules

In spectroscopy and atomic physics, the rate of transitions between states $|n\rangle$ and $|m\rangle$ induced by an oscillating electric field $\mathbf{E}(t)$ in the dipole approximation is proportional to the **electric dipole matrix element**:

$$\mu_{mn} \propto \langle m | \hat{x} | n \rangle$$

Using the ladder operator identity $\hat{x} = \sqrt{\frac{\hbar}{2m\omega}}(\hat{a} + \hat{a}^\dagger)$:

$$\langle m | \hat{x} | n \rangle = \sqrt{\frac{\hbar}{2m\omega}} \left( \langle m | \hat{a} | n \rangle + \langle m | \hat{a}^\dagger | n \rangle \right)$$

Evaluating the action of the ladder operators:

$$\langle m | \hat{x} | n \rangle = \sqrt{\frac{\hbar}{2m\omega}} \left( \sqrt{n}\,\delta_{m, n-1} + \sqrt{n+1}\,\delta_{m, n+1} \right)$$

This yields the fundamental **harmonic optical selection rule**:

$$\Delta n = m - n = \pm 1$$

A harmonic oscillator can **only absorb or emit a single quantum of energy $\hbar\omega$ at a time**! Transitions with $\Delta n = \pm 2, \pm 3, \dots$ are strictly forbidden.


In [ ]:
# Compute position matrix elements numerically using continuous wavefunctions:
# X_mn = \int psi_m^*(x) * x * psi_n(x) dx
N_levels = 6
X_matrix_num = np.zeros((N_levels, N_levels))

for m in range(N_levels):
    for n in range(N_levels):
        integrand = wavefuncs_num[m] * x_grid * wavefuncs_num[n]
        X_matrix_num[m, n] = integrate.simpson(y=integrand, x=x_grid)

# Compute theoretical matrix using ladder operators
try:
    X_op_qutip = np.sqrt(HBAR / (2.0 * MASS * OMEGA)) * (a + a_dag)
    X_matrix_exact = X_op_qutip.full()[:N_levels, :N_levels]
except Exception:
    X_matrix_exact = np.sqrt(HBAR / (2.0 * MASS * OMEGA)) * (a_mat + adag_mat)[:N_levels, :N_levels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

im1 = ax1.imshow(np.abs(X_matrix_num), cmap="YlGnBu", vmin=0, vmax=1.8)
ax1.set_title(r"Numerical Dipole Matrix $|\langle m | \hat{x} | n \rangle|$ (Continuous)")
ax1.set_xlabel("State Index $n$")
ax1.set_ylabel("State Index $m$")
for (i, j), val in np.ndenumerate(X_matrix_num):
    ax1.text(j, i, f"{val:.2f}", ha="center", va="center", color="white" if abs(val) > 0.8 else "black", fontsize=9)

im2 = ax2.imshow(np.abs(X_matrix_exact), cmap="YlGnBu", vmin=0, vmax=1.8)
ax2.set_title(r"Algebraic Matrix $\sqrt{\hbar/2m\omega}(\hat{a} + \hat{a}^\dagger)$")
ax2.set_xlabel("State Index $n$")
ax2.set_ylabel("State Index $m$")
for (i, j), val in np.ndenumerate(X_matrix_exact):
    ax2.text(j, i, f"{val:.2f}", ha="center", va="center", color="white" if abs(val) > 0.8 else "black", fontsize=9)

plt.tight_layout()
plt.show()

print("Notice that ONLY the immediate subdiagonal (m = n-1) and superdiagonal (m = n+1) are non-zero!")
print("This proves the strict dipole selection rule Delta n = +/- 1.")


---
## 6. The Bohr Correspondence Principle: Quantum vs. Classical Limit

Niels Bohr's **Correspondence Principle** (1920) states that in the limit of large quantum numbers ($n \gg 1$), quantum mechanical predictions must converge to classical mechanics.

* **Classically:** A particle oscillating with energy $E = \frac{1}{2}m\omega^2 A^2$ spends the majority of its time near the **turning points** ($x = \pm A$) where its velocity drops to zero ($v = dx/dt \to 0$). The classical probability density is:
  $$P_{\text{classical}}(x) = \frac{1}{\pi \sqrt{A^2 - x^2}} \quad (|x| < A)$$
* **Quantum Ground State ($n=0$):** The particle is most likely to be found at the **origin $x = 0$**, where the classical velocity is maximal!
* **High Excited States ($n \ge 12$):** Rapid quantum oscillations average out, and the envelope peaks dramatically at the turning points, perfectly matching the classical distribution!


In [ ]:
try:
    from ipywidgets import interact, IntSlider
    has_widgets = True
except ImportError:
    has_widgets = False

def plot_correspondence(n=12):
    # Solve on a wider grid for higher excited states
    L_wide = 8.0
    x_w, E_w, psi_w, _ = solve_continuous_harmonic_oscillator(L=L_wide, N=700)
    
    psi_n = psi_w[n]
    En = E_w[n]
    
    # Classical amplitude A = sqrt(2 * En / (m * omega^2))
    A_classical = np.sqrt(2.0 * En / (MASS * OMEGA**2))
    
    # Classical probability density
    P_classical = np.zeros_like(x_w)
    inside = np.abs(x_w) < (A_classical * 0.995)
    P_classical[inside] = 1.0 / (np.pi * np.sqrt(A_classical**2 - x_w[inside]**2))
    
    plt.figure(figsize=(10, 5))
    plt.plot(x_w, np.abs(psi_n)**2, color="#1f77b4", lw=1.8, label=rf"Quantum $|\psi_{{{n}}}(x)|^2$ ($n={n}$)")
    plt.plot(x_w, P_classical, color="#d62728", ls="--", lw=2.2, label=r"Classical $P_{\mathrm{classical}}(x) = \frac{1}{\pi\sqrt{A^2-x^2}}$")
    plt.axvline(A_classical, color="black", ls=":", label=rf"Classical Turning Points $x = \pm {A_classical:.2f}$")
    plt.axvline(-A_classical, color="black", ls=":")
    
    plt.title(rf"Bohr Correspondence Principle: Quantum vs. Classical ($n = {n}$, $E_{{{n}}} = {En:.1f}\hbar\omega$)")
    plt.xlabel("Position $x$")
    plt.ylabel("Probability Density")
    plt.xlim(-L_wide, L_wide)
    plt.ylim(0, max(0.4, np.max(np.abs(psi_n)**2) * 1.3))
    plt.legend(loc="upper center", framealpha=0.9)
    plt.show()

if has_widgets:
    interact(plot_correspondence, n=IntSlider(min=0, max=20, step=1, value=12, description="Quantum n:"));
else:
    print("Interactive widgets available in Jupyter/Colab. Displaying n = 12:")
    plot_correspondence(n=12)


---
## 7. Hands-On Exercises & Guided Challenges

### Exercise 1: Anharmonicity & The Kerr Nonlinearity
In nonlinear quantum optics, intense laser fields interact with nonlinear optical crystals (such as $\chi^{(3)}$ media), adding an anharmonic quartic potential:

$$\hat{H}' = \hat{H}_0 + \lambda \hat{x}^4$$

Using first-order perturbation theory, the energy shift for state $|n\rangle$ is:
$$\Delta E_n = \lambda \langle n | \hat{x}^4 | n \rangle$$

1. Express $\hat{x}^4$ in terms of ladder operators $\hat{a}, \hat{a}^\dagger$ and compute $\langle n | \hat{x}^4 | n \rangle$.
2. In QuTiP, construct $\hat{H} = \hbar\omega(\hat{a}^\dagger\hat{a} + 1/2) + \lambda \hat{x}^4$ with $\lambda = 0.05$ and find the new energy levels via `.eigenenergies()`. Is the energy level spacing still constant?

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Analytical result:**
Using $\hat{x} = \sqrt{\frac{\hbar}{2m\omega}}(\hat{a} + \hat{a}^\dagger)$, normal ordering yields:
$$\langle n | (\hat{a} + \hat{a}^\dagger)^4 | n \rangle = 6n^2 + 6n + 3$$
$$\Delta E_n = \lambda \left(\frac{\hbar}{2m\omega}\right)^2 (6n^2 + 6n + 3)$$
Because $\Delta E_n$ depends on $n^2$, **the energy levels are no longer equally spaced**! This is the physical mechanism behind the **Kerr optical effect** and **photon blockades** (Module 3 and 4).

**Code snippet:**
```python
x_op_q = np.sqrt(1.0/2.0) * (a + a_dag)
H_kerr = H_osc + 0.05 * (x_op_q**4)
energies_kerr = H_kerr.eigenenergies()
spacings = np.diff(energies_kerr[:5])
print("Energy level spacings E_{n+1} - E_n with Kerr nonlinearity:")
print(np.round(spacings, 4))
```
</details>

---

### Exercise 2: Spatial Parity Symmetry of Wavefunctions
The harmonic potential $V(x) = \frac{1}{2}m\omega^2 x^2$ is an even function of $x$ ($V(-x) = V(x)$).
1. Prove that $\psi_n(-x) = (-1)^n \psi_n(x)$ (even parity for even $n$, odd parity for odd $n$).
2. Explain why $\langle n | \hat{x} | n \rangle = 0$ for all $n$ based purely on parity arguments.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Explanation:**
The parity operator $\hat{\Pi}$ commutes with $\hat{H}$ ($[\hat{\Pi}, \hat{H}] = 0$). Since the eigenstates are non-degenerate, every eigenstate must be an eigenstate of parity with eigenvalue $+1$ or $-1$:
$$\psi_n(-x) = (-1)^n \psi_n(x)$$
For the expectation value $\langle n | \hat{x} | n \rangle = \int_{-\infty}^\infty |\psi_n(x)|^2 x \, dx$:
* The probability density $|\psi_n(x)|^2$ is strictly **even**.
* The position operator $x$ is strictly **odd**.
* The integrand is an odd function integrated over a symmetric interval $[-\infty, \infty]$, which is identically zero!
</details>


---
## 8. Summary & What's Next

### Key Takeaways
1. **Continuous vs. Discrete Formulations:** The continuous Schrödinger differential equation and Dirac's algebraic ladder operator formulation yield identical physics: $E_n = \hbar\omega(n + 1/2)$.
2. **Vacuum Energy:** The zero-point energy $E_0 = \frac{1}{2}\hbar\omega$ is an unavoidable consequence of the uncertainty principle ($\Delta x \Delta p \ge \hbar/2$).
3. **The Harmonic Oscillator IS Quantized Light:** Each electromagnetic cavity mode is a harmonic oscillator. Energy quanta $n$ are **photons**; coordinate $x$ maps to electric field $E$; momentum $p$ maps to magnetic field $B$.
4. **Dipole Selection Rules:** The linear coupling $\hat{x} \propto (\hat{a} + \hat{a}^\dagger)$ restricts dipole transitions to $\Delta n = \pm 1$.
5. **Correspondence Principle:** For $n \gg 1$, quantum probabilities peak at the classical turning points, reproducing classical mechanics in the macroscopic limit.

---

### What's Next?
In **Notebook 0.5 (`05_time_evolution_and_wavepackets.ipynb`)**, we will:
* Simulate time evolution under the Schrödinger equation using the **Split-Operator Fourier Method (SOFM)** and matrix exponentiation.
* Compare the dispersive spreading of a free particle against the **non-spreading coherent oscillation** of a displaced wavepacket in a harmonic potential.
* Complete Module 0 and bridge directly into **Module 1 (Quantum States of Light)**!

---
### References & Recommended Reading
* **David J. Griffiths**, *Introduction to Quantum Mechanics*, Chapter 2: Time-Independent Schrödinger Equation (The Harmonic Oscillator).
* **Christopher Gerry & Peter Knight**, *Introductory Quantum Optics*, Chapter 1 & 2: Quantization of the Electromagnetic Field.
* **Claude Cohen-Tannoudji, Jacques Dupont-Roc, Gilbert Grynberg**, *Photons and Atoms: Introduction to Quantum Electrodynamics*.
